In [1]:
import os, glob, pprint
base = "/kaggle/input/datasets/challabhagyadeepthi/uav-dataset"
dataset_entries = []
for d in os.listdir(base):
    p = os.path.join(base, d)
    if os.path.isdir(p):
        files = []
        for root, _, filenames in os.walk(p):
            for f in filenames:
                files.append(os.path.join(root, f))
        dataset_entries.append((d, files[:50]))  # show up to 50 files per dataset
pprint.pprint(dataset_entries)
print("\nIf you uploaded two zip files, expect to see something like '/kaggle/input/<your-dataset>/<something>.zip'")

[('labels-20240430T212029Z-001',
  ['/kaggle/input/datasets/challabhagyadeepthi/uav-dataset/labels-20240430T212029Z-001/labels/val.cache',
   '/kaggle/input/datasets/challabhagyadeepthi/uav-dataset/labels-20240430T212029Z-001/labels/test.cache',
   '/kaggle/input/datasets/challabhagyadeepthi/uav-dataset/labels-20240430T212029Z-001/labels/train.cache',
   '/kaggle/input/datasets/challabhagyadeepthi/uav-dataset/labels-20240430T212029Z-001/labels/val/f450_1757.txt',
   '/kaggle/input/datasets/challabhagyadeepthi/uav-dataset/labels-20240430T212029Z-001/labels/val/multi_1453.txt',
   '/kaggle/input/datasets/challabhagyadeepthi/uav-dataset/labels-20240430T212029Z-001/labels/val/f450_1220.txt',
   '/kaggle/input/datasets/challabhagyadeepthi/uav-dataset/labels-20240430T212029Z-001/labels/val/phantom_1587.txt',
   '/kaggle/input/datasets/challabhagyadeepthi/uav-dataset/labels-20240430T212029Z-001/labels/val/f450_1087.txt',
   '/kaggle/input/datasets/challabhagyadeepthi/uav-dataset/labels-202404

In [2]:
%%bash
set -e

KINPUT="/kaggle/input/datasets/challabhagyadeepthi/uav-dataset"
WORK="/kaggle/working/dataset"
mkdir -p "$WORK"

# pick the first dataset folder under /kaggle/input/uav-dataset (usually images-... and labels-...)
FIRST_DATASET="$(ls "$KINPUT" | head -n1)"
echo "Using dataset folder: $FIRST_DATASET"

# copy everything from that folder into working (safe copy)
cp -r "$KINPUT/$FIRST_DATASET"/* "$WORK/" || true

# unzip any zip files found (handles nested zips too)
for z in "$WORK"/*.zip "$WORK"/*/*.zip; do
  if [ -f "$z" ]; then
    echo "Unzipping $z"
    unzip -q -o "$z" -d "$WORK"
  fi
done

echo
echo "Final /kaggle/working/dataset contents:"
ls -la "$WORK"
echo
echo "Recursive listing (first 200 lines):"
ls -R "$WORK" | sed -n '1,200p'

Using dataset folder: images-20240430T215401Z-001

Final /kaggle/working/dataset contents:
total 12
drwxr-xr-x 3 root root 4096 Mar 12 15:11 .
drwxr-xr-x 3 root root 4096 Mar 12 15:11 ..
drwxr-xr-x 5 root root 4096 Mar 12 15:12 images

Recursive listing (first 200 lines):
/kaggle/working/dataset:
images

/kaggle/working/dataset/images:
test
train
val

/kaggle/working/dataset/images/test:
f450_107.jpg
f450_113.jpg
f450_126.jpg
f450_127.jpg
f450_128.jpg
f450_14.jpg
f450_17.jpg
f450_21.jpg
f450_28.jpg
f450_29.jpg
f450_32.jpg
f450_40.jpg
f450_44.jpg
f450_54.jpg
f450_6.jpg
f450_7.jpg
f450_85.jpg
f450_8.jpg
f450_92.jpg
f450_93.jpg

/kaggle/working/dataset/images/train:
background_0.jpg
background_100.jpg
background_101.jpg
background_102.jpg
background_103.jpg
background_105.jpg
background_106.jpg
background_107.jpg
background_108.jpg
background_109.jpg
background_10.jpg
background_110.jpg
background_111.jpg
background_112.jpg
background_113.jpg
background_114.jpg
background_115.jpg
backgrou

In [3]:
%%bash
set -e

WORK="/kaggle/working/dataset"

echo "Copying LABELS..."
cp -r /kaggle/input/datasets/challabhagyadeepthi/uav-dataset/labels-20240430T212029Z-001/labels "$WORK/"

echo "Now the dataset contains:"
ls -R "$WORK"


Copying LABELS...
Now the dataset contains:
/kaggle/working/dataset:
images
labels

/kaggle/working/dataset/images:
test
train
val

/kaggle/working/dataset/images/test:
f450_107.jpg
f450_113.jpg
f450_126.jpg
f450_127.jpg
f450_128.jpg
f450_14.jpg
f450_17.jpg
f450_21.jpg
f450_28.jpg
f450_29.jpg
f450_32.jpg
f450_40.jpg
f450_44.jpg
f450_54.jpg
f450_6.jpg
f450_7.jpg
f450_85.jpg
f450_8.jpg
f450_92.jpg
f450_93.jpg

/kaggle/working/dataset/images/train:
background_0.jpg
background_100.jpg
background_101.jpg
background_102.jpg
background_103.jpg
background_105.jpg
background_106.jpg
background_107.jpg
background_108.jpg
background_109.jpg
background_10.jpg
background_110.jpg
background_111.jpg
background_112.jpg
background_113.jpg
background_114.jpg
background_115.jpg
background_116.jpg
background_118.jpg
background_119.jpg
background_11.jpg
background_120.jpg
background_121.jpg
background_122.jpg
background_124.jpg
background_125.jpg
background_126.jpg
background_127.jpg
background_12.jpg
back

In [4]:
data_yaml = """path: /kaggle/working/dataset/images
train: train
val: val
test: test
nc: 6
names: ['f450', 'helicopter', 'hushan', 'multi', 'phantom', 'plane']
"""

dst = "/kaggle/working/dataset/data.yaml"
with open(dst, "w") as f:
    f.write(data_yaml)
print("Wrote", dst)
print(open(dst).read())

Wrote /kaggle/working/dataset/data.yaml
path: /kaggle/working/dataset/images
train: train
val: val
test: test
nc: 6
names: ['f450', 'helicopter', 'hushan', 'multi', 'phantom', 'plane']



In [5]:
!yolo val \
model=/kaggle/input/datasets/challabhagyadeepthi/yolo26-pt-model/yolo26n.pt \
data=/kaggle/working/dataset/data.yaml \
imgsz=640 \
batch=8 \
project=/kaggle/working/eval \
name=val_yolo26n \
save_json=True \
plots=True

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2238.5±1314.5 MB/s, size: 240.1 KB)
val: Scanning /kaggle/working/dataset/labels/val... 1520 images, 497 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1520/1520 1.2Kit/s 1.3s
val: New cache created: /kaggle/working/dataset/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 190/190 12.7it/s 15.0s
                   all       1520       1146     0.0405      0.266     0.0341      0.018
                pe

In [6]:
!yolo train model=/kaggle/input/datasets/challabhagyadeepthi/yolo26-pt-model/yolo26n.pt data=/kaggle/working/dataset/data.yaml epochs=10 imgsz=640 batch=16 project=/kaggle/working/runs name=drone_finetune

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/datasets/challabhagyadeepthi/yolo26-pt-model/yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=drone_finetune, nbs=64, nms=False, opset=None, optim

In [7]:
!yolo val \
model=/kaggle/working/runs/drone_finetune/weights/best.pt \
data=/kaggle/working/dataset/data.yaml \
imgsz=640 \
batch=8 \
save_json=True \
plots=True \
project=/kaggle/working/eval \
name=val_best

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2572.6±1259.9 MB/s, size: 220.6 KB)
val: Scanning /kaggle/working/dataset/labels/val.cache... 1520 images, 497 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1520/1520 193.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 190/190 15.3it/s 12.4s
                   all       1520       1146      0.957      0.921      0.976      0.771
                  f450        294        294      0.982      0.963      0.993      0.859
            helicopter        424        425       0.94      0.877      0.953      0.695
                hushan        425        427      0.949      0.924      0.982      0.759
Speed: 0.6ms preprocess, 3.0ms inference, 0.0ms loss, 0.2ms postprocess per image
Saving /kaggle/working/eval/val_best/

In [8]:
with open("/kaggle/working/test_data.yaml", "w") as f:
    f.write("""
path: /kaggle/working/test_dataset

train: images/test
val: images/test
test: images/test

nc: 6
names: ['f450', 'helicopter', 'hushan', 'multi', 'phantom', 'plane']
""")

In [9]:
!mkdir -p /kaggle/working/test_dataset/images/test
!mkdir -p /kaggle/working/test_dataset/labels/test

In [10]:
!cp "/kaggle/input/datasets/challabhagyadeepthi/test-labels/test labels/"* \
   /kaggle/working/test_dataset/labels/test/

!cp -r /kaggle/input/datasets/challabhagyadeepthi/uav-dataset/images-20240430T215401Z-001/images/test \
   /kaggle/working/test_dataset/images/

In [11]:
!yolo val \
model=/kaggle/working/runs/drone_finetune/weights/best.pt \
data=/kaggle/working/test_data.yaml \
split=test \
imgsz=640 \
batch=8 \
save_json=True \
plots=True \
project=/kaggle/working/eval \
name=test_eval

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2204.8±232.7 MB/s, size: 290.5 KB)
val: Scanning /kaggle/working/test_dataset/labels/test... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 804.3it/s 0.0s
val: New cache created: /kaggle/working/test_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 5.1it/s 0.6s
                   all         20         20      0.994          1      0.995      0.752
                  f450         20         20      0.994          1      0.995      0.752
Speed: 3.3ms preprocess, 11.8ms inference, 0.0ms loss, 0.2ms postprocess per image
Saving /kaggle/working/eval/test_eval/predictions.json...
Results saved to /kaggle/working/eval/test_eval
💡 Learn more at https://docs.ultralytics.com/modes/va

In [12]:
!yt-dlp -f best -o "video.mp4" "https://www.youtube.com/watch?v=jdvLV6dYyRU"

         To let yt-dlp download and merge the best available formats, simply do not pass any format selection.
         If you know what you are doing and want only the best pre-merged format, use "-f b" instead to suppress this warning
[youtube] Extracting URL: https://www.youtube.com/watch?v=jdvLV6dYyRU
[youtube] jdvLV6dYyRU: Downloading webpage
[youtube] jdvLV6dYyRU: Downloading android vr player API JSON
[info] jdvLV6dYyRU: Downloading 1 format(s): 18
[download] Destination: video.mp4



In [13]:
import cv2
import os

video_path = "video.mp4"
output_folder = "frames"
os.makedirs(output_folder, exist_ok=True)

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
print("Video FPS:", fps)

frame_count = 0
save_count = 0

# Extract 1 frame per second
frame_interval = 1

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_count % frame_interval == 0:
        filename = os.path.join(output_folder, f"frame_{save_count:05d}.jpg")
        cv2.imwrite(filename, frame)
        save_count += 1

    frame_count += 1

cap.release()
print("Frames extracted:", save_count)

Video FPS: 30.0
Frames extracted: 2143


In [14]:
from ultralytics import YOLO

model26 = YOLO("/kaggle/working/runs/drone_finetune/weights/best.pt ")


In [15]:
import cv2
import os
from ultralytics import YOLO


frames_path = "frames"
frame_files = sorted(os.listdir(frames_path))

all_detections_26 = []


for frame_name in frame_files:
    frame_path = os.path.join(frames_path, frame_name)
    frame = cv2.imread(frame_path)

    # Run YOLOv8
    results26 = model26(frame)[0]
    detections26 = []
    for box in results26.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0])
        cls = int(box.cls[0])
        detections26.append([x1, y1, x2, y2, conf, cls])
    all_detections_26.append(detections26)

    

print("Detection completed ")


0: 384x640 1 f450, 75.2ms
Speed: 3.3ms preprocess, 75.2ms inference, 24.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 f450, 14.3ms
Speed: 3.1ms preprocess, 14.3ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 f450, 12.2ms
Speed: 1.8ms preprocess, 12.2ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 f450, 12.2ms
Speed: 2.2ms preprocess, 12.2ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 f450, 13.1ms
Speed: 1.5ms preprocess, 13.1ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 f450s, 13.4ms
Speed: 2.0ms preprocess, 13.4ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 f450, 12.0ms
Speed: 1.4ms preprocess, 12.0ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 f450, 12.3ms
Speed: 2.4ms preprocess, 12.3ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 38

In [16]:

import numpy as np

class KalmanFilter:
    def __init__(self):

        dt = 1

        self.F = np.array([
            [1,0,0,0,dt,0],
            [0,1,0,0,0,dt],
            [0,0,1,0,0,0],
            [0,0,0,1,0,0],
            [0,0,0,0,1,0],
            [0,0,0,0,0,1]
        ])

        self.H = np.array([
            [1,0,0,0,0,0],
            [0,1,0,0,0,0],
            [0,0,1,0,0,0],
            [0,0,0,1,0,0]
        ])

        self.P = np.eye(6) * 10
        self.Q = np.eye(6) * 0.01
        self.R = np.eye(4) * 1

    def predict(self, x):
        x = self.F @ x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return x

    def update(self, x, z):
        y = z - self.H @ x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)

        x = x + K @ y
        self.P = (np.eye(6) - K @ self.H) @ self.P
        return x


In [17]:

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2-x1) * max(0, y2-y1)

    area1 = (box1[2]-box1[0])*(box1[3]-box1[1])
    area2 = (box2[2]-box2[0])*(box2[3]-box2[1])

    union = area1 + area2 - inter

    return inter / union if union > 0 else 0



In [18]:
class Track:
    def __init__(self, bbox, track_id):
        self.kf = KalmanFilter()
        self.id = track_id
        cx = (bbox[0]+bbox[2])/2
        cy = (bbox[1]+bbox[3])/2
        w = bbox[2]-bbox[0]
        h = bbox[3]-bbox[1]

        self.state = np.array([cx,cy,w,h,0,0])

    def predict(self):
        self.state = self.kf.predict(self.state)

    def update(self, bbox):
        cx = (bbox[0]+bbox[2])/2
        cy = (bbox[1]+bbox[3])/2
        w = bbox[2]-bbox[0]
        h = bbox[3]-bbox[1]

        z = np.array([cx,cy,w,h])
        self.state = self.kf.update(self.state, z)



In [19]:
import numpy as np
import cv2
import os

def kalman_prediction_accuracy_visual(
        all_detections,
        frame_files,
        frames_path,
        output_folder="kalman_pred_frames"):

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    tracks = []
    next_id = 0
    iou_threshold = 0.3
    prediction_ious = []

    num_frames = min(len(all_detections), len(frame_files))
    print(f"🚀 Evaluating Kalman motion on {num_frames} frames...")

    for t in range(num_frames - 1):

        current_dets = all_detections[t]
        next_dets = all_detections[t+1]

        # ---------- LOAD NEXT FRAME (for drawing) ----------
        frame_path = os.path.join(frames_path, frame_files[t+1])
        frame = cv2.imread(frame_path)
        if frame is None:
            continue

        # ---------- UPDATE TRACKS USING CURRENT DETECTIONS ----------
        updated_tracks = []

        for det in current_dets:
            best_iou, best_track = 0, None

            for track in tracks:
                cx, cy, w, h = track.state[:4]
                bbox = [cx-w/2, cy-h/2, cx+w/2, cy+h/2]
                score = iou(det[:4], bbox)

                if score > best_iou:
                    best_iou, best_track = score, track

            if best_iou > iou_threshold:
                best_track.update(det[:4])
                updated_tracks.append(best_track)
            else:
                updated_tracks.append(Track(det[:4], next_id))
                next_id += 1

        tracks = updated_tracks

        # ---------- PREDICT NEXT POSITIONS ----------
        predicted_boxes = []
        for track in tracks:
            track.predict()
            cx, cy, w, h = track.state[:4]
            pred_bbox = [cx-w/2, cy-h/2, cx+w/2, cy+h/2]
            predicted_boxes.append(pred_bbox)

        # ---------- DRAW ACTUAL DETECTIONS (RED) ----------
        for det in next_dets:
            x1, y1, x2, y2 = map(int, det[:4])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)

        # ---------- DRAW PREDICTIONS (BLUE) ----------
        for pred in predicted_boxes:
            x1,y1,x2,y2 = map(int, pred)
            cv2.rectangle(frame, (x1,y1), (x2,y2), (255,0,0), 2)

        # ---------- LEGEND ----------
        legend_x = 20
        legend_y = 30

        # Detection (Red)
        cv2.rectangle(frame, (legend_x, legend_y-15), (legend_x+20, legend_y+5), (0,0,255), -1)
        cv2.putText(frame, "Tracking", (legend_x+30, legend_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

        # Prediction (Blue)
        cv2.rectangle(frame, (legend_x, legend_y+20-15), (legend_x+20, legend_y+20+5), (255,0,0), -1)
        cv2.putText(frame, "Detection", (legend_x+30, legend_y+20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

        # ---------- COMPUTE IOU ACCURACY ----------
        for pred in predicted_boxes:
            best_iou = 0
            for det in next_dets:
                score = iou(pred, det[:4])
                if score > best_iou:
                    best_iou = score
            prediction_ious.append(best_iou)

        # ---------- SAVE FRAME ----------
        cv2.imwrite(os.path.join(output_folder, f"frame_{t:04d}.jpg"), frame)

    # ---------- METRICS ----------
    avg_iou = np.mean(prediction_ious) if prediction_ious else 0
    success_rate = sum(i > 0.5 for i in prediction_ious) / len(prediction_ious) if prediction_ious else 0

    print("\n📊 Kalman Motion Prediction Accuracy")
    print(f"Average IOU: {avg_iou:.3f}")
    print(f"Success Rate (IOU>0.5): {success_rate:.3f}")

    return avg_iou, success_rate

In [20]:
avg_iou26, success26 = kalman_prediction_accuracy_visual(
    all_detections_26, 
    frame_files, 
    frames_path,
    output_folder='tracking_results_26'
)


🚀 Evaluating Kalman motion on 2143 frames...

📊 Kalman Motion Prediction Accuracy
Average IOU: 0.804
Success Rate (IOU>0.5): 0.965
